In [2]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.utils.data as Data
import torch.nn.functional as F

from google.colab import drive
from tqdm.auto import trange


drive.mount('/content/drive')


BASE_DIR = '/content/drive/MyDrive/Insider_Threat_Project'
SAMPLE_DATA_DIR = os.path.join(BASE_DIR, 'sample_data')
RESULT_DIR = os.path.join(BASE_DIR, 'lstm_feature_extraction_result')

BEST_MODEL_DIR = os.path.join(RESULT_DIR, 'early_stop_model')
CHECKPOINT_DIR = os.path.join(RESULT_DIR, 'checkpoints')
PLOT_DIR = os.path.join(RESULT_DIR, 'plots')
LOG_DIR = os.path.join(RESULT_DIR, 'logs')

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)


SEQUENCE_FILE = os.path.join(
    SAMPLE_DATA_DIR,
    '1-data-test-combine-sequence.csv'
)

MANUAL_FILE = os.path.join(
    SAMPLE_DATA_DIR,
    '1-data-test-combine.csv'
)


if not os.path.exists(SEQUENCE_FILE):
    raise FileNotFoundError(
        f'Cannot find sequence CSV: {SEQUENCE_FILE}'
    )

if not os.path.exists(MANUAL_FILE):
    raise FileNotFoundError(
        f'Cannot find manual-feature CSV: {MANUAL_FILE}'
    )


SEQUENCE_LENGTH = 74
INPUT_SIZE = 25
HIDDEN_SIZE = 100
ENCODE_SIZE = 5
N_FEATURES = 5

EPOCHS = 1000
LEARNING_RATE = 0.01
TRAIN_ROUNDS = 5
BATCH_SIZE = 2000

PATIENCE = 20
DELTA = 0.0001

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print('Device:', DEVICE)

if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))


sequence_data = pd.read_csv(SEQUENCE_FILE)
df_manual = pd.read_csv(MANUAL_FILE)

print('Sequence data shape:', sequence_data.shape)
print('Manual feature data shape:', df_manual.shape)


if len(sequence_data) != len(df_manual):
    raise ValueError(
        'The two CSV files have different row counts. '
        'Do not concatenate their features until their rows match.'
    )


identity_columns = [
    'date_index',
    'user_index',
    'date',
    'label'
]

for col in identity_columns:
    if col not in sequence_data.columns:
        raise ValueError(
            f'{col} is missing from the sequence CSV.'
        )

    if col not in df_manual.columns:
        raise ValueError(
            f'{col} is missing from the manual feature CSV.'
        )


sequence_identity = (
    sequence_data[identity_columns]
    .fillna('')
    .astype(str)
    .reset_index(drop=True)
)

manual_identity = (
    df_manual[identity_columns]
    .fillna('')
    .astype(str)
    .reset_index(drop=True)
)

if not sequence_identity.equals(manual_identity):
    raise ValueError(
        'The rows of the sequence and manual-feature CSVs '
        'do not match in the same order.'
    )

print('Row identity and ordering verified.')


if len(sequence_data.columns) < 4 + SEQUENCE_LENGTH:
    raise ValueError(
        'The sequence CSV does not contain 74 activity columns '
        'after its first four metadata columns.'
    )


activity_columns = list(
    sequence_data.columns[4:4 + SEQUENCE_LENGTH]
)

sequence_data[activity_columns] = (
    sequence_data[activity_columns].fillna(0)
)

activity_values = sequence_data[activity_columns].to_numpy(
    dtype=np.float32
)

if not np.isfinite(activity_values).all():
    raise ValueError(
        'The activity sequences contain infinite or invalid values.'
    )

if not np.allclose(
    activity_values,
    np.round(activity_values)
):
    raise ValueError(
        'Activity values must be integers for one-hot encoding.'
    )

activity_values = np.round(activity_values).astype(np.int64)

if activity_values.min() < 0 or activity_values.max() >= INPUT_SIZE:
    raise ValueError(
        f'Activity values must be between 0 and {INPUT_SIZE - 1}.'
    )


x = torch.tensor(
    activity_values,
    dtype=torch.long
)

x = F.one_hot(
    x,
    num_classes=INPUT_SIZE
).float()

labels = sequence_data['label'].to_numpy(dtype=np.int64)

y = torch.tensor(labels, dtype=torch.int32)

print('Encoded sequence shape:', tuple(x.shape))
print('Labels shape:', tuple(y.shape))
print('Label counts:')
print(pd.Series(labels).value_counts().sort_index())


batch_data = Data.DataLoader(
    dataset=Data.TensorDataset(x, y),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)


class LstmAutoEncoder(nn.Module):

    def __init__(
        self,
        input_size=25,
        hidden_size=100,
        batch_size=2000,
        encode_size=5
    ):
        super(LstmAutoEncoder, self).__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.batch_size = batch_size
        self.encode_size = encode_size

        self.encoder_lstm = nn.LSTM(
            self.input_size,
            self.hidden_size,
            batch_first=True
        )

        self.encoder_fc = nn.Linear(
            self.hidden_size,
            self.encode_size
        )

        self.decoder_lstm = nn.LSTM(
            self.hidden_size,
            self.input_size,
            batch_first=True
        )

        self.decoder_fc = nn.Linear(
            self.encode_size,
            self.hidden_size
        )

        self.relu = nn.ReLU()

        self.output_fc = nn.Linear(
            SEQUENCE_LENGTH * self.encode_size,
            N_FEATURES
        )

        self.decoder_input_fc = nn.Linear(
            N_FEATURES,
            SEQUENCE_LENGTH * self.encode_size
        )

    def forward(self, x):

        encoder_lstm, (n, c) = self.encoder_lstm(x)

        encoder_fc = self.encoder_fc(encoder_lstm)

        encoder_out = encoder_fc.flatten(1)

        encoder_out = self.output_fc(encoder_out)

        decoder_input = self.decoder_input_fc(encoder_out)

        decoder_input = decoder_input.view(
            -1,
            SEQUENCE_LENGTH,
            self.encode_size
        )

        decoder_fc = self.decoder_fc(decoder_input)

        decoder_lstm, (n, c) = self.decoder_lstm(decoder_fc)

        return encoder_out, decoder_lstm


class EarlyStopping:

    def __init__(
        self,
        save_path,
        patience=20,
        verbose=True,
        delta=0.0001,
        metric='loss'
    ):
        self.save_path = save_path
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = float('inf')
        self.delta = delta
        self.metric = metric
        self.val_bound = float('inf')
        self.sign = -1 if metric == 'loss' else 1

    def __call__(self, value, model):

        value = float(value)
        score = value * self.sign

        if self.best_score is None:

            self.best_score = score
            self.save_checkpoint(value, model)

        elif score < self.best_score + self.delta:

            self.counter += 1

            print(
                f'EarlyStopping counter: '
                f'{self.counter} out of {self.patience}'
            )

            if self.counter >= self.patience:
                self.early_stop = True

        else:

            self.best_score = score
            self.save_checkpoint(value, model)
            self.counter = 0

    def save_checkpoint(self, value, model):

        if self.verbose:

            if self.sign == -1:
                print(
                    f'Validation {self.metric} decreased '
                    f'({self.val_bound:.6f} --> {value:.6f}). '
                    f'Saving model ...'
                )
            else:
                print(
                    f'Validation {self.metric} increased '
                    f'({self.val_bound:.6f} --> {value:.6f}). '
                    f'Saving model ...'
                )

        torch.save(
            model.state_dict(),
            self.save_path
        )

        self.val_bound = value


def save_training_checkpoint(
    path,
    model,
    optimizer,
    epoch,
    early_stopping,
    train_losses,
    validation_losses,
    round_number,
    finished=False
):

    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': epoch,
        'round_number': round_number,
        'early_stopping': {
            'counter': early_stopping.counter,
            'best_score': early_stopping.best_score,
            'early_stop': early_stopping.early_stop,
            'val_loss_min': early_stopping.val_loss_min,
            'val_bound': early_stopping.val_bound
        },
        'train_losses': train_losses,
        'validation_losses': validation_losses,
        'finished': finished,
        'torch_rng_state': torch.get_rng_state(),
        'numpy_rng_state': np.random.get_state(),
        'python_rng_state': random.getstate()
    }

    if torch.cuda.is_available():
        checkpoint['cuda_rng_state_all'] = (
            torch.cuda.get_rng_state_all()
        )

    temporary_path = path + '.tmp'

    torch.save(checkpoint, temporary_path)

    os.replace(temporary_path, path)


def load_training_checkpoint(
    path,
    model,
    optimizer,
    early_stopping
):

    checkpoint = torch.load(
        path,
        map_location=DEVICE,
        weights_only=False
    )

    model.load_state_dict(
        checkpoint['model_state_dict']
    )

    optimizer.load_state_dict(
        checkpoint['optimizer_state_dict']
    )

    state = checkpoint['early_stopping']

    early_stopping.counter = state['counter']
    early_stopping.best_score = state['best_score']
    early_stopping.early_stop = state['early_stop']
    early_stopping.val_loss_min = state['val_loss_min']
    early_stopping.val_bound = state['val_bound']

    torch.set_rng_state(
        checkpoint['torch_rng_state'].cpu()
    )

    np.random.set_state(
        checkpoint['numpy_rng_state']
    )

    random.setstate(
        checkpoint['python_rng_state']
    )

    if (
        torch.cuda.is_available()
        and 'cuda_rng_state_all' in checkpoint
    ):
        torch.cuda.set_rng_state_all(
            checkpoint['cuda_rng_state_all']
        )

    return checkpoint


def save_loss_plot(
    train_losses,
    validation_losses,
    output_path
):

    plt.figure(figsize=(9, 5))

    epochs_completed = range(1, len(train_losses) + 1)

    plt.plot(
        epochs_completed,
        train_losses,
        label='Training loss'
    )

    plt.plot(
        epochs_completed,
        validation_losses,
        label='Monitoring loss'
    )

    if validation_losses:
        best_epoch = int(
            np.argmin(validation_losses)
        ) + 1

        plt.axvline(
            best_epoch,
            linestyle='--',
            label='Lowest monitoring loss'
        )

    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title('LSTM Autoencoder Training')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        output_path,
        dpi=200
    )

    plt.close()


start_time = time.time()

round_results = []

for r in range(TRAIN_ROUNDS):

    print('\n' + '=' * 60)
    print(f'TRAINING ROUND {r}')
    print('=' * 60)

    best_model_path = os.path.join(
        BEST_MODEL_DIR,
        f'seq{SEQUENCE_LENGTH}_{r}round_best.pt'
    )

    checkpoint_path = os.path.join(
        CHECKPOINT_DIR,
        f'seq{SEQUENCE_LENGTH}_{r}round_latest.pt'
    )

    complete_path = os.path.join(
        LOG_DIR,
        f'seq{SEQUENCE_LENGTH}_{r}round_complete.json'
    )

    history_path = os.path.join(
        LOG_DIR,
        f'seq{SEQUENCE_LENGTH}_{r}round_history.csv'
    )

    plot_path = os.path.join(
        PLOT_DIR,
        f'seq{SEQUENCE_LENGTH}_{r}round_loss.png'
    )

    if os.path.exists(complete_path):

        with open(complete_path, 'r') as f:
            completed_result = json.load(f)

        if os.path.exists(best_model_path):

            print(
                f'Round {r} was already completed. '
                'Loading its saved results.'
            )

            round_results.append(completed_result)
            continue

    model = LstmAutoEncoder(
        input_size=INPUT_SIZE,
        hidden_size=HIDDEN_SIZE,
        batch_size=BATCH_SIZE,
        encode_size=ENCODE_SIZE
    ).to(DEVICE)

    loss_function = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    early_stopping = EarlyStopping(
        save_path=best_model_path,
        verbose=True,
        patience=PATIENCE,
        delta=DELTA,
        metric='loss'
    )

    train_losses = []
    validation_losses = []
    start_epoch = 0

    if os.path.exists(checkpoint_path):

        print('Found a previous checkpoint.')
        print('Resuming the interrupted round.')

        checkpoint = load_training_checkpoint(
            checkpoint_path,
            model,
            optimizer,
            early_stopping
        )

        train_losses = checkpoint['train_losses']
        validation_losses = checkpoint['validation_losses']

        start_epoch = checkpoint['epoch']

        if checkpoint.get('finished', False):

            print('This round was already marked as finished.')

            if os.path.exists(best_model_path):

                result = {
                    'round': r,
                    'min_loss': float(min(validation_losses)),
                    'stop_epoch': len(validation_losses)
                }

                with open(complete_path, 'w') as f:
                    json.dump(result, f)

                round_results.append(result)
                continue

    for epoch in trange(
        start_epoch,
        EPOCHS,
        desc=f'Round {r}'
    ):

        model.train()

        epoch_batch_losses = []

        for b_x, b_y in batch_data:

            b_x = b_x.to(DEVICE)

            optimizer.zero_grad()

            _, y_pred = model(b_x)

            loss = loss_function(
                y_pred,
                b_x
            )

            loss.backward()

            optimizer.step()

            epoch_batch_losses.append(
                loss.item()
            )

        train_loss = float(
            epoch_batch_losses[-1]
        )

        train_losses.append(train_loss)

        model.eval()

        monitoring_data = x[-500:].to(DEVICE)

        with torch.no_grad():

            _, monitoring_prediction = model(
                monitoring_data
            )

            monitoring_loss = loss_function(
                monitoring_prediction,
                monitoring_data
            ).item()

        validation_losses.append(
            float(monitoring_loss)
        )

        early_stopping(
            monitoring_loss,
            model
        )

        current_epoch = epoch + 1

        save_training_checkpoint(
            checkpoint_path,
            model,
            optimizer,
            current_epoch,
            early_stopping,
            train_losses,
            validation_losses,
            r,
            finished=early_stopping.early_stop
        )

        if current_epoch % 10 == 0:

            print(
                f'Round {r} | '
                f'Epoch {current_epoch} | '
                f'Train loss: {train_loss:.6f} | '
                f'Monitoring loss: {monitoring_loss:.6f}'
            )

        if early_stopping.early_stop:

            print(
                f'Early stopping at epoch {current_epoch}'
            )

            break

    save_loss_plot(
        train_losses,
        validation_losses,
        plot_path
    )

    history_df = pd.DataFrame({
        'epoch': range(1, len(train_losses) + 1),
        'training_loss': train_losses,
        'monitoring_loss': validation_losses
    })

    history_df.to_csv(
        history_path,
        index=False
    )

    result = {
        'round': r,
        'min_loss': float(min(validation_losses)),
        'stop_epoch': len(validation_losses)
    }

    with open(complete_path, 'w') as f:
        json.dump(result, f)

    round_results.append(result)

    print(f'Round {r} completed.')
    print(f'Lowest monitoring loss: {result["min_loss"]:.6f}')
    print(f'Epochs completed: {result["stop_epoch"]}')
    print(f'Best model saved at: {best_model_path}')


perform = pd.DataFrame(round_results)

perform = perform.sort_values(
    'round'
).reset_index(drop=True)

perform.to_csv(
    os.path.join(
        RESULT_DIR,
        f'perform_result_seq{SEQUENCE_LENGTH}.csv'
    ),
    index=False
)

print('\nResults of all training rounds:')
print(perform)

best_round = int(
    perform.loc[
        perform['min_loss'].idxmin(),
        'round'
    ]
)

best_model_path = os.path.join(
    BEST_MODEL_DIR,
    f'seq{SEQUENCE_LENGTH}_{best_round}round_best.pt'
)

print('\nBest round:', best_round)
print('Best model path:', best_model_path)


model_extract = LstmAutoEncoder(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE,
    batch_size=BATCH_SIZE,
    encode_size=ENCODE_SIZE
).to(DEVICE)

model_extract.load_state_dict(
    torch.load(
        best_model_path,
        map_location=DEVICE,
        weights_only=True
    )
)

model_extract.eval()

with torch.no_grad():

    output, _ = model_extract(
        x.to(DEVICE)
    )

output = output.cpu().numpy()

df_feature = pd.DataFrame(
    output,
    columns=[f'lstm_feature_{i + 1}' for i in range(N_FEATURES)]
)

if len(df_feature) != len(df_manual):
    raise ValueError(
        'Extracted feature rows do not match manual-feature rows.'
    )

df_save = pd.concat(
    [
        df_manual.reset_index(drop=True),
        df_feature.reset_index(drop=True)
    ],
    axis=1
)

feature_path = os.path.join(
    RESULT_DIR,
    f'1-data-test-manual+lstm_seq{SEQUENCE_LENGTH}.csv'
)

df_save.to_csv(
    feature_path,
    index=False
)

print('\nFeature extraction completed.')
print('Extracted LSTM feature shape:', df_feature.shape)
print('Combined feature shape:', df_save.shape)
print('Combined features saved to:', feature_path)

print('\nTotal time:', round(time.time() - start_time, 2), 'seconds')
print('\nAll results are stored in Google Drive.')

Mounted at /content/drive
Device: cuda
GPU: Tesla T4
Sequence data shape: (1908, 103)
Manual feature data shape: (1908, 10)
Row identity and ordering verified.
Encoded sequence shape: (1908, 74, 25)
Labels shape: (1908,)
Label counts:
0    954
1    954
Name: count, dtype: int64

TRAINING ROUND 0


Round 0:   0%|          | 0/1000 [00:00<?, ?it/s]

Validation loss decreased (inf --> 0.033186). Saving model ...
Validation loss decreased (0.033186 --> 0.023974). Saving model ...
Validation loss decreased (0.023974 --> 0.017393). Saving model ...
Validation loss decreased (0.017393 --> 0.015509). Saving model ...
Validation loss decreased (0.015509 --> 0.014492). Saving model ...
Validation loss decreased (0.014492 --> 0.013901). Saving model ...
Validation loss decreased (0.013901 --> 0.013568). Saving model ...
Validation loss decreased (0.013568 --> 0.013367). Saving model ...
Validation loss decreased (0.013367 --> 0.013194). Saving model ...
Validation loss decreased (0.013194 --> 0.012948). Saving model ...
Round 0 | Epoch 10 | Train loss: 0.014276 | Monitoring loss: 0.012948
Validation loss decreased (0.012948 --> 0.012494). Saving model ...
Validation loss decreased (0.012494 --> 0.011764). Saving model ...
EarlyStopping counter: 1 out of 20
Validation loss decreased (0.011764 --> 0.011081). Saving model ...
Validation loss 

Round 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Validation loss decreased (inf --> 0.025742). Saving model ...
Validation loss decreased (0.025742 --> 0.017739). Saving model ...
Validation loss decreased (0.017739 --> 0.015172). Saving model ...
Validation loss decreased (0.015172 --> 0.014442). Saving model ...
Validation loss decreased (0.014442 --> 0.014166). Saving model ...
Validation loss decreased (0.014166 --> 0.013937). Saving model ...
Validation loss decreased (0.013937 --> 0.013627). Saving model ...
Validation loss decreased (0.013627 --> 0.013179). Saving model ...
EarlyStopping counter: 1 out of 20
Validation loss decreased (0.013179 --> 0.012091). Saving model ...
Round 1 | Epoch 10 | Train loss: 0.014037 | Monitoring loss: 0.012091
Validation loss decreased (0.012091 --> 0.011759). Saving model ...
Validation loss decreased (0.011759 --> 0.010989). Saving model ...
EarlyStopping counter: 1 out of 20
Validation loss decreased (0.010989 --> 0.010560). Saving model ...
Validation loss decreased (0.010560 --> 0.009948)

Round 2:   0%|          | 0/1000 [00:00<?, ?it/s]

Validation loss decreased (inf --> 0.040189). Saving model ...
Validation loss decreased (0.040189 --> 0.031286). Saving model ...
Validation loss decreased (0.031286 --> 0.025969). Saving model ...
Validation loss decreased (0.025969 --> 0.021997). Saving model ...
Validation loss decreased (0.021997 --> 0.018325). Saving model ...
Validation loss decreased (0.018325 --> 0.015380). Saving model ...
Validation loss decreased (0.015380 --> 0.013407). Saving model ...
Validation loss decreased (0.013407 --> 0.012330). Saving model ...
Validation loss decreased (0.012330 --> 0.011683). Saving model ...
Validation loss decreased (0.011683 --> 0.011187). Saving model ...
Round 2 | Epoch 10 | Train loss: 0.012573 | Monitoring loss: 0.011187
Validation loss decreased (0.011187 --> 0.010781). Saving model ...
Validation loss decreased (0.010781 --> 0.010369). Saving model ...
Validation loss decreased (0.010369 --> 0.010056). Saving model ...
EarlyStopping counter: 1 out of 20
Validation loss 

Round 3:   0%|          | 0/1000 [00:00<?, ?it/s]

Validation loss decreased (inf --> 0.029655). Saving model ...
Validation loss decreased (0.029655 --> 0.020299). Saving model ...
Validation loss decreased (0.020299 --> 0.015623). Saving model ...
Validation loss decreased (0.015623 --> 0.014954). Saving model ...
EarlyStopping counter: 1 out of 20
EarlyStopping counter: 2 out of 20
Validation loss decreased (0.014954 --> 0.014728). Saving model ...
Validation loss decreased (0.014728 --> 0.014585). Saving model ...
Validation loss decreased (0.014585 --> 0.014459). Saving model ...
Validation loss decreased (0.014459 --> 0.014353). Saving model ...
Round 3 | Epoch 10 | Train loss: 0.015664 | Monitoring loss: 0.014353
EarlyStopping counter: 1 out of 20
Validation loss decreased (0.014353 --> 0.014179). Saving model ...
EarlyStopping counter: 1 out of 20
Validation loss decreased (0.014179 --> 0.014006). Saving model ...
Validation loss decreased (0.014006 --> 0.013905). Saving model ...
Validation loss decreased (0.013905 --> 0.01378

Round 4:   0%|          | 0/1000 [00:00<?, ?it/s]

Validation loss decreased (inf --> 0.046450). Saving model ...
Validation loss decreased (0.046450 --> 0.038694). Saving model ...
Validation loss decreased (0.038694 --> 0.035048). Saving model ...
Validation loss decreased (0.035048 --> 0.032782). Saving model ...
Validation loss decreased (0.032782 --> 0.030210). Saving model ...
Validation loss decreased (0.030210 --> 0.027015). Saving model ...
Validation loss decreased (0.027015 --> 0.023040). Saving model ...
Validation loss decreased (0.023040 --> 0.020523). Saving model ...
Validation loss decreased (0.020523 --> 0.016438). Saving model ...
Validation loss decreased (0.016438 --> 0.014673). Saving model ...
Round 4 | Epoch 10 | Train loss: 0.017054 | Monitoring loss: 0.014673
Validation loss decreased (0.014673 --> 0.013430). Saving model ...
Validation loss decreased (0.013430 --> 0.012644). Saving model ...
Validation loss decreased (0.012644 --> 0.012200). Saving model ...
Validation loss decreased (0.012200 --> 0.011903). 